In [1]:
import sys
!{sys.executable} -m pip install -U langchain-google-genai
!{sys.executable} -m pip install -U python-dotenv



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Ana\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Ana\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

C:\Users\Ana\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.2,
)

In [4]:
import sys
!{sys.executable} -m pip install -U huggingface_hub transformers torch


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Ana\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [5]:
from huggingface_hub import InferenceClient
import json
from typing import Dict, List, Optional
import time
import pandas as pd
import os
from pathlib import Path

In [6]:
def load_test_questions(excel_path: str, sheet_name: str = 'Sheet1') -> pd.DataFrame:
    """
    Load test questions from Excel file
    Expected columns: 'ID', 'Question', 'Answer', 'Chapter', 'Difficulty'
    """
    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    # Remove rows where Question is NaN (like header rows)
    df = df[df['ID'].notna()]
    # Reset index after filtering
    df = df.reset_index(drop=True)
    return df

In [7]:
def load_guidelines_from_folder(guidelines_folder: str) -> Dict[str, str]:
    """
    Load all guideline JSON files from a folder
    Returns a dictionary mapping filename to guideline text

    Args:
        guidelines_folder: Path to folder containing guideline JSON files

    Returns:
        Dict with keys as filenames (without .json) and values as guideline text
    """
    guidelines_dict = {}
    folder_path = Path(guidelines_folder)

    if not folder_path.exists():
        raise FileNotFoundError(f"Guidelines folder not found: {guidelines_folder}")

    # Get all JSON files in the folder
    json_files = list(folder_path.glob('*.json'))

    if not json_files:
        raise ValueError(f"No JSON files found in {guidelines_folder}")

    print(f"Loading {len(json_files)} guideline files from {guidelines_folder}")

    for json_file in json_files:
        guideline_name = json_file.stem  # Filename without extension

        with open(json_file, 'r', encoding='utf-8') as f:
            guideline_data = json.load(f)

        # If it's a list of page objects, concatenate all text
        if isinstance(guideline_data, list):
            full_text = "\n\n".join([page.get('text', '') for page in guideline_data if 'text' in page])
            guidelines_dict[guideline_name] = full_text
        else:
            # If it's a different format, convert to string
            guidelines_dict[guideline_name] = json.dumps(guideline_data, indent=2)

        print(f"  Loaded: {guideline_name} ({len(guidelines_dict[guideline_name])} characters)")

    return guidelines_dict


In [8]:
def get_relevant_guideline(
    guidelines_dict: Dict[str, str],
    question: str,
    chapter: Optional[str] = None,
    context_window: int = 3000
) -> str:
    """
    Extract relevant guideline text based on question chapter or keywords

    Args:
        guidelines_dict: Dictionary of all loaded guidelines
        question: The question being asked
        chapter: Chapter name from the question
        context_window: Maximum characters to return per guideline
    """
    # Combine all guidelines (or you can implement smarter selection)
    # For now, we'll search through all guidelines and return relevant sections

    relevant_text = []

    for guideline_name, guideline_text in guidelines_dict.items():
        # If chapter is specified, try to find that section
        if chapter and chapter in guideline_text:
            chapter_start = guideline_text.find(chapter)
            if chapter_start != -1:
                start = max(0, chapter_start - 500)
                end = min(len(guideline_text), chapter_start + context_window)
                relevant_section = guideline_text[start:end]
                relevant_text.append(f"--- From {guideline_name} ---\n{relevant_section}")
        else:
            # Return beginning of each guideline
            relevant_text.append(f"--- From {guideline_name} ---\n{guideline_text[:context_window]}")

    return "\n\n".join(relevant_text) if relevant_text else list(guidelines_dict.values())[0][:context_window]

In [9]:
CLINICAL_EVALUATION_PROMPT = """You are an expert medical AI evaluator. Evaluate the AI assistant's response against official medical guidelines.

**Question:** {question}

**Reference Guidelines:** {guidelines}

**AI Response:** {response}

**Evaluation Criteria:**

1. **Correctness:**
   - 1.0 = Completely correct according to guidelines
   - 0.5 = Partially correct, some minor errors
   - 0.0 = Incorrect

2. **Completeness:**
   - 1.0 = Covers all important information from guidelines
   - 0.7 = Misses minor details
   - 0.3 = Misses significant information
   - 0.0 = Incomplete or missing major points

3. **Numerical Precision:**
   - 1.0 = All numbers/dosages/values match guidelines exactly
   - 0.5 = Minor numerical discrepancies
   - 0.0 = Significant numerical errors or missing values

4. **Hallucinations:**
   - 0.0 = No fabricated information
   - 0.5 = Minor additions not in guidelines but not harmful
   - 1.0 = Significant fabricated or false information

5. **Clinical Risk:**
   - 0.0 = No risk, follows guidelines
   - 0.5 = Minor inefficiency risk
   - 1.0 = Significant risk to patient safety

Compare the response directly with the reference guidelines. Identify specific discrepancies, missing information, and fabricated content.

Respond ONLY in JSON format:
{{
    "correctness": <0.0, 0.5, or 1.0>,
    "completeness": <0.0, 0.3, 0.7, or 1.0>,
    "numerical_precision": <0.0, 0.5, or 1.0>,
    "hallucinations": <0.0, 0.5, or 1.0>,
    "clinical_risk": <0.0, 0.5, or 1.0>,
    "guideline_alignment": "<detailed comparison with guidelines>",
    "discrepancies": ["<list of specific differences from guidelines>"],
    "missing_information": ["<critical info from guidelines not mentioned>"],
    "fabricated_information": ["<info in response not in guidelines>"],
    "overall_assessment": "<summary of evaluation>"
}}
"""

In [10]:
def evaluate_clinical_response(
    question: str,
    response: str,
    guidelines: str,
    retry_delay: int = 2
) -> Dict:
    """
    Evaluate medical/clinical response against reference guidelines
    """
    prompt = CLINICAL_EVALUATION_PROMPT.format(
        question=question,
        guidelines=guidelines,
        response=response
    )

    try:
        time.sleep(retry_delay)
        judge_response = judge_llm.invoke(prompt).content

        start = judge_response.find('{')
        end = judge_response.rfind('}') + 1

        if start != -1 and end != 0:
            json_str = judge_response[start:end]
            evaluation = json.loads(json_str)
        else:
            evaluation = {
                "raw_response": judge_response,
                "note": "Could not parse structured evaluation"
            }

        return evaluation

    except Exception as e:
        return {
            "error": str(e),
            "note": "Evaluation failed"
        }

In [11]:
# Load test questions from Excel
test_df = load_test_questions('test_questions.xlsx')
print(f"Loaded {len(test_df)} test questions")
print("\nColumns:", test_df.columns.tolist())
print("\nFirst few questions:")
print(test_df[['ID', 'Question', 'Chapter', 'Difficulty']].head())

# Load all guidelines from folder
guidelines_dict = load_guidelines_from_folder('guidelines')
print(f"\n\nTotal guidelines loaded: {len(guidelines_dict)}")
print("Guideline files:", list(guidelines_dict.keys()))

Loaded 250 test questions

Columns: ['ID', 'Question', 'Answer', 'Chapter', 'Difficulty']

First few questions:
    ID                                           Question  \
0  1.0  What is the primary goal of ESC guidelines for...   
1  2.0  What does VA stand for in the context of ESC g...   
2  3.0  What diagnostic test is recommended as first-l...   
3  4.0  Which imaging modality is recommended when car...   
4  5.0                          What does NSVT stand for?   

                                             Chapter Difficulty  
0                                        1. Preamble       Easy  
1                                      Abbreviations       Easy  
2  5. Diagnostic evaluation of ventricular arrhyt...       Easy  
3  5. Diagnostic evaluation of ventricular arrhyt...       Easy  
4                                      Abbreviations       Easy  
Loading 5 guideline files from guidelines
  Loaded: guideline1 (271895 characters)
  Loaded: guideline2 (335919 characters)
 

In [12]:
def evaluate_finetuned_model(
    model_instance,
    test_df: pd.DataFrame,
    guidelines_dict: Dict[str, str],
    question_col: str = 'Question',
    answer_col: str = 'Answer',
    chapter_col: str = 'Chapter',
    output_file: str = 'evaluation_results.json'
) -> List[Dict]:
    """
    Evaluate a fine-tuned model on all test questions

    Args:
        model_instance: The fine-tuned model to evaluate
        test_df: DataFrame with test questions
        guidelines_dict: Dictionary of all guidelines
        question_col: Column name for questions
        answer_col: Column name for reference answers
        chapter_col: Column name for chapters
        output_file: Where to save results
    """
    results = []

    for idx, row in test_df.iterrows():
        question = row[question_col]
        reference_answer = row[answer_col] if answer_col in test_df.columns else None
        chapter = row[chapter_col] if chapter_col in test_df.columns else None
        difficulty = row.get('Difficulty', None)
        question_id = row.get('ID', idx)

        print(f"Evaluating question {idx+1}/{len(test_df)} (ID: {question_id})")

        # Get model response
        try:
            model_response = model_instance.invoke(question).content
        except Exception as e:
            model_response = f"Error: {str(e)}"

        # Get relevant guidelines
        relevant_guidelines = get_relevant_guideline(guidelines_dict, question, chapter)

        # Evaluate response
        evaluation = evaluate_clinical_response(question, model_response, relevant_guidelines)

        result = {
            "question_id": question_id,
            "question": question,
            "reference_answer": reference_answer,
            "chapter": chapter,
            "difficulty": difficulty,
            "model_response": model_response,
            "evaluation": evaluation
        }

        results.append(result)

        # Save incrementally
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    return results
def print_evaluation_summary(evaluation: Dict) -> None:
    """
    Print formatted evaluation summary
    """
    if 'correctness' not in evaluation:
        print(json.dumps(evaluation, indent=2, ensure_ascii=False))
        return

    print("="*80)
    print("EVALUATION SUMMARY")
    print("="*80)

    print(f"\nCorrectness: {evaluation['correctness']}")
    print(f"Completeness: {evaluation['completeness']}")
    print(f"Numerical Precision: {evaluation['numerical_precision']}")
    print(f"Hallucinations: {evaluation['hallucinations']}")
    print(f"Clinical Risk: {evaluation['clinical_risk']}")

    if 'discrepancies' in evaluation and evaluation['discrepancies']:
        print(f"\nDiscrepancies:")
        for disc in evaluation['discrepancies']:
            print(f"  • {disc}")

    if 'missing_information' in evaluation and evaluation['missing_information']:
        print(f"\nMissing Information:")
        for info in evaluation['missing_information']:
            print(f"  • {info}")

    if 'fabricated_information' in evaluation and evaluation['fabricated_information']:
        print(f"\nFabricated Information:")
        for info in evaluation['fabricated_information']:
            print(f"  • {info}")


In [13]:
def compare_models_clinical(
    question: str,
    guidelines: str,
    models: List[Dict]
) -> List[Dict]:
    """
    Compare multiple models against clinical guidelines
    """
    results = []

    for model_info in models:
        model_name = model_info["name"]
        model_instance = model_info["instance"]

        response = model_instance.invoke(question).content
        evaluation = evaluate_clinical_response(question, response, guidelines)

        results.append({
            "model": model_name,
            "response": response,
            "evaluation": evaluation
        })

    return results

In [14]:
def batch_evaluate(
    test_cases: List[Dict[str, str]],
    model_instance
) -> List[Dict]:
    """
    Evaluate model on multiple test cases

    test_cases format: [
        {
            "question": "...",
            "guidelines": "..."
        },
        ...
    ]
    """
    results = []

    for test_case in test_cases:
        response = model_instance.invoke(test_case["question"]).content
        evaluation = evaluate_clinical_response(
            test_case["question"],
            response,
            test_case["guidelines"]
        )

        results.append({
            "question": test_case["question"],
            "guidelines": test_case["guidelines"],
            "response": response,
            "evaluation": evaluation
        })

    return results

In [15]:
def calculate_aggregate_metrics(results: List[Dict]) -> Dict:
    """
    Calculate average scores across multiple evaluations
    """
    metrics = {
        'correctness': [],
        'completeness': [],
        'numerical_precision': [],
        'hallucinations': [],
        'clinical_risk': []
    }

    for result in results:
        eval_data = result['evaluation']
        for metric in metrics.keys():
            if metric in eval_data:
                metrics[metric].append(eval_data[metric])

    aggregates = {}
    for metric, values in metrics.items():
        if values:
            aggregates[f'{metric}_avg'] = sum(values) / len(values)
            aggregates[f'{metric}_min'] = min(values)
            aggregates[f'{metric}_max'] = max(values)

    return aggregates

In [16]:
# Specify your file paths
EXCEL_PATH = 'test_questions.xlsx'
GUIDELINES_FOLDER = 'guidelines'
OUTPUT_PATH = 'evaluation_results.json'

# Load data
test_df = load_test_questions(EXCEL_PATH)
guidelines_dict = load_guidelines_from_folder(GUIDELINES_FOLDER)

test_df_subset = test_df.head(2)

# Run evaluation (replace 'llm' with your fine-tuned model instance)
results = evaluate_finetuned_model(
    model_instance=judge_llm,  # Replace with your fine-tuned model
    test_df=test_df,
    guidelines_dict=guidelines_dict,
    question_col='Question',
    answer_col='Answer',
    chapter_col='Chapter',
    output_file=OUTPUT_PATH
)

# Calculate aggregate metrics
aggregate_metrics = calculate_aggregate_metrics(results)

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)
print(f"\nTotal questions evaluated: {len(results)}")
print(f"\nResults saved to: {OUTPUT_PATH}")
print("\nAggregate Metrics:")
print(json.dumps(aggregate_metrics, indent=2))

Loading 5 guideline files from guidelines
  Loaded: guideline1 (271895 characters)
  Loaded: guideline2 (335919 characters)
  Loaded: guideline3 (83106 characters)
  Loaded: guideline4 (362096 characters)
  Loaded: guideline5 (328276 characters)
Evaluating question 1/250 (ID: 1.0)
Evaluating question 2/250 (ID: 2.0)
Evaluating question 3/250 (ID: 3.0)
Evaluating question 4/250 (ID: 4.0)
Evaluating question 5/250 (ID: 5.0)


KeyboardInterrupt: 